<a href="https://colab.research.google.com/github/raheelarif86/AI_Training_November25/blob/main/My_Version_ENEC_rag_vector_stores_pineconeindexdemo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pinecone Vector Store

If you're opening this Notebook on colab, you will probably need to install LlamaIndex 🦙.

In [ ]:
%pip install llama-index llama-index-vector-stores-pinecone

In [ ]:
import logging
import sys
import os

logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logging.getLogger().addHandler(logging.StreamHandler(stream=sys.stdout))

#### Creating a Pinecone Index

In [ ]:
from pinecone import Pinecone, ServerlessSpec

In [ ]:
import openai
from google.colab import userdata

# Retrieve the OpenAI API key from Google Colab secrets
openai.api_key = userdata.get('openai')

if openai.api_key:
    os.environ["OPENAI_API_KEY"] = openai.api_key

In [ ]:
import os
from google.colab import userdata
from pinecone import Pinecone, ServerlessSpec # Added import statement

api_key = userdata.get('PINECONE_API_KEY')

if api_key:
    os.environ["PINECONE_API_KEY"] = api_key

pc = Pinecone(api_key=api_key)

In [ ]:
api_key

In [ ]:
# dimensions are for text-embedding-ada-002

index_name = "quickstart"
# Delete index if it already exists
if index_name in [idx["name"] for idx in pc.list_indexes()]:
    pc.delete_index(index_name)

pc.create_index(
    name=index_name,
    dimension=1536,
    metric="euclidean",
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)

In [ ]:
pinecone_index = pc.Index("quickstart")

#### Load documents, build the PineconeVectorStore and VectorStoreIndex

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.vector_stores.pinecone import PineconeVectorStore
from IPython.display import Markdown, display

Download Data

In [ ]:
import os
import requests

# Create the 'data' directory if it doesn't exist
if not os.path.exists("data"):
    os.makedirs("data")

# Define the URL of the document to download
document_url = "https://raw.githubusercontent.com/raheelarif86/AI_Training_November25/main/data/SOP_%20Troubleshooting%20Overheated%20Boiler%20Feed%20Pump%20(Pump%20P-201).pdf"
document_path = "./data/SOP_ Troubleshooting Overheated Boiler Feed Pump (Pump P-201).pdf"

# Download the document if it doesn't exist locally
if not os.path.exists(document_path):
    print(f"Downloading {document_path}...")
    response = requests.get(document_url)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
    with open(document_path, "wb") as f:
        f.write(response.content)
    print("Download complete.")
else:
    print(f"{document_path} already exists. Skipping download.")

# load documents
documents = SimpleDirectoryReader("./data").load_data()

In [ ]:
documents

The previous error occurred because the variable `pinecone_index` was used before it was defined. To fix this, I have combined the code that defines `pinecone_index` and the code that uses it into a single cell.

In [ ]:
# Initialize Pinecone index
pinecone_index = pc.Index("quickstart")

# initialize without metadata filter
from llama_index.core import StorageContext

vector_store = PineconeVectorStore(pinecone_index=pinecone_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context
)

#### Query Index

May take a minute or so for the index to be ready!

In [ ]:
# set Logging to DEBUG for more detailed outputs
import time

query_engine = index.as_query_engine()
start_time = time.time()
response = query_engine.query("what is this document about")
display(Markdown(f"<b>{response}</b>"))

# End timer and print duration
end_time = time.time()
print(f"\nExecution Time: {end_time - start_time:.2f} seconds")

In [ ]:
display(Markdown(f"<b>{response}</b>"))

In [ ]:
pip install gradio

In [ ]:
!pip install llama-index llama-index-vector-stores-pinecone openai

# --- Imports ---
import os
import logging
import sys
import gradio as gr
from IPython.display import Markdown, display
import requests # Import requests for downloading files

from pinecone import Pinecone, ServerlessSpec
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext, Settings # Added Settings import
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.llms.openai import OpenAI

# --- Initialize Pinecone ---

index_name = "quickstart"
dimension = 1536

# Initialize Pinecone client (assuming API key is set in environment or Colab secrets)
api_key = os.environ.get('PINECONE_API_KEY')
if not api_key:
    from google.colab import userdata
    api_key = userdata.get('PINECONE_API_KEY')
    os.environ["PINECONE_API_KEY"] = api_key
pc = Pinecone(api_key=api_key)


# Create Pinecone index
if index_name not in [idx["name"] for idx in pc.list_indexes()]:
    print(f"Creating Pinecone index: {index_name}")
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="euclidean",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
else:
    print(f"Pinecone index: {index_name} already exists.")

pinecone_index = pc.Index(index_name)

# --- Load Data ---
# Create folders & download a sample doc
if not os.path.exists("data"):
    print("Creating directory 'data'...")
    os.makedirs("data")
else:
    print("Directory 'data' already exists.")

document_url = "https://raw.githubusercontent.com/raheelarif86/AI_Training_November25/main/data/SOP_%20Troubleshooting%20Overheated%20Boiler%20Feed%20Pump%20(Pump%20P-201).pdf"
document_path = "./data/SOP_ Troubleshooting Overheated Boiler Feed Pump (Pump P-201).pdf"

# Attempt to download the document if it doesn't exist
if not os.path.exists(document_path):
    print(f"Attempting to download document to {document_path} from {document_url}...")
    try:
        response = requests.get(document_url)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        with open(document_path, "wb") as f:
            f.write(response.content)
        print(f"Document downloaded successfully to {document_path}.")
    except requests.exceptions.RequestException as e:
        print(f"ERROR: Failed to download document. Reason: {e}")
        # If download fails, we must ensure SimpleDirectoryReader doesn't get called on an empty dir
        raise # Re-raise the exception to stop execution and prevent subsequent errors
else:
    print(f"Document {document_path} already exists. Skipping download.")

# Debugging: List contents of the directory before loading
print(f"Contents of directory './data': {os.listdir('./data')}")
if not os.listdir('./data'):
    raise ValueError("Directory './data' is empty after download check. Cannot proceed with SimpleDirectoryReader.")

documents = SimpleDirectoryReader("./data").load_data()

# --- Create Index ---
vector_store = PineconeVectorStore(pinecone_index=pinecone_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)

# --- System Prompt (polite + answer-from-document constraint) ---
SYSTEM_PROMPT = """You are Aisha, a polite and professional Insurance assistant.
Answer ONLY using the information found in the indexed insurance document(s).
If the answer is not in the document(s), say: "I couldn’t find that in the document."
Keep responses concise, helpful, and courteous.
"""

# Ensure OpenAI API key is set for LlamaIndex Settings
openai_api_key = os.environ.get('OPENAI_API_KEY')
if not openai_api_key:
    from google.colab import userdata
    openai_api_key = userdata.get('openai')
    if openai_api_key:
        os.environ["OPENAI_API_KEY"] = openai_api_key
    else:
        raise ValueError("OPENAI_API_KEY not found in environment or Colab secrets.")

Settings.llm = OpenAI(model="gpt-5.2-2025-12-11", temperature=0)

# --- Query Engine ---
query_engine = index.as_query_engine()

def query_doc(user_question: str):
    if not user_question or not user_question.strip():
        return "Please enter a question."
    full_query = f"""{SYSTEM_PROMPT}

User question:
{user_question.strip()}
"""
    try:
        response = query_engine.query(full_query)
        text = str(response).strip()
        # Gentle post-processing to keep it brief/polite
        return text if text else "I couldn’t find that in the document."
    except Exception as e:
        return f"Error: {str(e)}"

# --- Gradio UI (Professional look with logo, centered title) ---
# Use the raw GitHub URL for proper image rendering.
LOGO_URL = "https://github.com/raheelarif86/AI_Training_November25/blob/main/Enec_Logo.png?raw=true"

CUSTOM_CSS = """
.gradio-container { font-family: Inter, ui-sans-serif, system-ui, -apple-system, Segoe UI, Roboto, 'Helvetica Neue', Arial; }
.header-wrap {
    display: grid;
    grid-template-columns: 120px 1fr 120px;
    align-items: center;
    gap: 12px;
    padding: 12px 0 8px;
    border-bottom: 1px solid #eaeaea;
}
.header-logo { display:flex; align-items:center; justify-content:flex-start; }
.header-logo img { height: 48px; object-fit: contain; }
.header-title { text-align:center; }
.header-title h1 {
    margin: 0; font-weight: 700; font-size: 1.5rem; line-height: 1.2;
}
.header-spacer { height: 1px; }
.section { padding-top: 8px; }
.footer-note { text-align:center; font-size: 12px; color:#667085; padding: 8px 0 0; }
label.svelte-1ipelgc, .label-wrap label { font-weight: 600; }
"""

with gr.Blocks(css=CUSTOM_CSS, title="Insurance Document QA (LlamaIndex + Pinecone)") as demo:
    # Header with logo (left) and centered title
    with gr.Row(elem_classes="header-wrap"):
        with gr.Column(scale=0, elem_classes="header-logo"):
            gr.HTML(f'<img src="{LOGO_URL}" alt="Omantel Logo" />')
        with gr.Column(scale=1, elem_classes="header-title"):
            gr.HTML("<h1>ENEC</h1>")
        with gr.Column(scale=0):
            gr.HTML("")  # right-side spacer

    gr.Markdown(
        "Ask questions based on the Insurance Document "
        "**Answers come only from the document**. If not found, I’ll say so."
    )

    with gr.Group(elem_classes="section"):
        inp = gr.Textbox(
            label="Your question",
            placeholder="e.g., Ask in Insurance Question?",
            lines=2,
        )
        btn = gr.Button("Submit", variant="primary")
        out = gr.Textbox(label="Answer", lines=8)

    btn.click(fn=query_doc, inputs=inp, outputs=out)
    inp.submit(fn=query_doc, inputs=inp, outputs=out)

    gr.Markdown('<div class="footer-note">LlamaIndex + Pinecone • Demo</div>')

demo.launch()

Replicate the Deplpyment hugging face
https://huggingface.co/spaces/decodingdatascience/ddsinsurance1
openai key


In [ ]:
documents